In [14]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [15]:
import os

print(os.path.exists("/content/drive/MyDrive/fl_data.pkl"))

True


In [16]:
from google.colab import drive
drive.mount('/content/drive', force_remount=True)

Mounted at /content/drive


In [17]:
import os

print("Drive mounted:", os.path.exists("/content/drive/MyDrive"))

print("Files in MyDrive:")
print(os.listdir("/content/drive/MyDrive")[:20])

Drive mounted: True
Files in MyDrive:
['Colab Notebooks', 'dataset', 'fl_data.pkl', 'binary_dataset.pkl']


In [18]:
import pickle

with open("/content/drive/MyDrive/binary_dataset.pkl", "rb") as f:
    data = pickle.load(f)

X_train = data["X_train"]
X_test = data["X_test"]
y_train = data["y_train"]
y_test = data["y_test"]
client_data = data["client_data"]
poisoned_client_data = data["poisoned_client_data"]
NUM_CLIENTS = data["NUM_CLIENTS"]

In [19]:
for var in [
    "X_train", "X_test", "y_train", "y_test",
    "client_data", "poisoned_client_data"
]:
    print(var, var in globals())

X_train True
X_test True
y_train True
y_test True
client_data True
poisoned_client_data True


In [20]:
import pickle

with open("/content/drive/MyDrive/binary_dataset.pkl", "wb") as f:
    pickle.dump({
        "X_train": X_train,
        "X_test": X_test,
        "y_train": y_train,
        "y_test": y_test,
        "client_data": client_data,
        "poisoned_client_data": poisoned_client_data,
        "NUM_CLIENTS": NUM_CLIENTS
    }, f)

print("Saved successfully.")

Saved successfully.


In [21]:
import os

print(os.path.exists("/content/drive/MyDrive/binary_dataset.pkl"))
print(os.path.getsize("/content/drive/MyDrive/binary_dataset.pkl"))

True
1772724675


In [22]:
from google.colab import drive
drive.mount('/content/drive')

import pickle

with open("/content/drive/MyDrive/binary_dataset.pkl", "rb") as f:
    data = pickle.load(f)

X_train = data["X_train"]
X_test = data["X_test"]
y_train = data["y_train"]
y_test = data["y_test"]
client_data = data["client_data"]
poisoned_client_data = data["poisoned_client_data"]
NUM_CLIENTS = data["NUM_CLIENTS"]

print("Loaded successfully")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Loaded successfully


In [33]:
import random

NUM_MALICIOUS = 50
random.seed(42)

malicious_ids = random.sample(range(NUM_CLIENTS), NUM_MALICIOUS)

poisoned_client_data = []

for cid, (X_client, y_client) in enumerate(client_data):
    if cid in malicious_ids:
        y_poison = 1 - y_client
        poisoned_client_data.append((X_client, y_poison))
    else:
        poisoned_client_data.append((X_client, y_client))

In [34]:
print("malicious_ids" in globals())

True


In [35]:
import pickle

with open("/content/drive/MyDrive/binary_dataset.pkl", "wb") as f:
    pickle.dump({
        "X_train": X_train,
        "X_test": X_test,
        "y_train": y_train,
        "y_test": y_test,
        "client_data": client_data,
        "poisoned_client_data": poisoned_client_data,
        "NUM_CLIENTS": NUM_CLIENTS,
        "malicious_ids": malicious_ids
    }, f)

print("Saved successfully.")

Saved successfully.


In [36]:
import pickle

with open("/content/drive/MyDrive/binary_dataset.pkl", "rb") as f:
    data = pickle.load(f)

print(data.keys())

dict_keys(['X_train', 'X_test', 'y_train', 'y_test', 'client_data', 'poisoned_client_data', 'NUM_CLIENTS', 'malicious_ids'])


In [23]:
import numpy as np
import random

from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Input

from sklearn.metrics import accuracy_score

In [24]:
def create_binary_model():

    model = Sequential([
        Input(shape=(46,)),
        Dense(256, activation="relu"),
        Dense(128, activation="relu"),
        Dense(64, activation="relu"),
        Dense(1, activation="sigmoid")
    ])

    model.compile(
        optimizer="adam",
        loss="binary_crossentropy",
        metrics=["accuracy"]
    )

    return model

global_model = create_binary_model()
global_weights = global_model.get_weights()

print("Global model ready")

Global model ready


In [25]:
def train_client(client_x, client_y, global_weights):

    local_model = create_binary_model()

    local_model.set_weights(global_weights)

    local_model.fit(
        client_x,
        client_y,
        epochs=1,
        batch_size=64,
        verbose=0
    )

    return local_model.get_weights()

In [26]:
def federated_average(local_weights):

    avg_weights = []

    for weights in zip(*local_weights):
        avg_weights.append(np.mean(weights, axis=0))

    return avg_weights

In [27]:
ROUNDS = 10
CLIENTS_PER_ROUND = 50

history = []

for rnd in range(ROUNDS):

    print(f"\nRound {rnd+1}")

    selected_clients = random.sample(
        poisoned_client_data,
        CLIENTS_PER_ROUND
    )

    local_weights = []

    for X_client, y_client in selected_clients:

        weights = train_client(
            X_client,
            y_client,
            global_weights
        )

        local_weights.append(weights)

    global_weights = federated_average(local_weights)

    global_model.set_weights(global_weights)

    pred = (
        global_model.predict(X_test, verbose=0) > 0.5
    ).astype(int)

    acc = accuracy_score(y_test, pred)

    history.append(acc)

    print("Accuracy =", round(acc,4))


Round 1
Accuracy = 0.9852

Round 2
Accuracy = 0.9831

Round 3
Accuracy = 0.9869

Round 4
Accuracy = 0.9879

Round 5
Accuracy = 0.9867

Round 6
Accuracy = 0.9875

Round 7
Accuracy = 0.9881

Round 8
Accuracy = 0.9885

Round 9
Accuracy = 0.9884

Round 10
Accuracy = 0.9883


In [28]:
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    confusion_matrix
)

pred = (
    global_model.predict(X_test, verbose=0) > 0.5
).astype(int)

print("Accuracy :", accuracy_score(y_test, pred))
print("Precision:", precision_score(y_test, pred))
print("Recall   :", recall_score(y_test, pred))
print("F1 Score :", f1_score(y_test, pred))

print(confusion_matrix(y_test, pred))

Accuracy : 0.9882982534220554
Precision: 0.9996000212036952
Recall   : 0.9884097518826013
F1 Score : 0.9939733921991128
[[ 15031    249]
 [  7297 622284]]
